Imports

In [3]:
import numpy as np
import gymnasium as gym
import h5py

Setup Environment and Extract Dataset

In [10]:
# Initialise data dictionary 
data_dict = {}

# Get dataset
with h5py.File("SafetyPointGoal2Gymnasium_v0_200_3442.hdf5", 'r') as f:
    print(list(f.keys()))
    for key in list(f.keys()):
        data_dict[key] = f[key][:]

print(list(data_dict.keys()))

['actions', 'costs', 'next_observations', 'observations', 'rewards', 'terminals', 'timeouts']
['actions', 'costs', 'next_observations', 'observations', 'rewards', 'terminals', 'timeouts']


Check its usable

In [11]:
violations = data_dict["costs"] > 0.0 # array of whole trajectory where each entry is 0 or 1
terminals = data_dict["terminals"].astype(bool)
timeouts = data_dict["timeouts"].astype(bool)

ends = np.where(terminals | timeouts)[0]
starts = np.concatenate([[0], ends[:-1] + 1])

keep = np.zeros(len(violations), dtype=bool)
new_terminal = np.zeros(len(violations), dtype=bool)
new_timeout = np.zeros(len(violations), dtype=bool)

for s, e in zip(starts, ends):
    traj = violations[s : e+1]

    if traj.any():
        k = s + int(np.argmax(traj))
        keep[s : k+1] = True
        new_terminal[k] = True
    else:
        keep[s : e+1] = True
        new_terminal[e] = terminals[e]
        new_timeout[e] = timeouts[e]

trunc_data_dict = {}
for key, val in data_dict.items():
    if isinstance(val, np.ndarray) and val.shape[0] == len(violations):
        trunc_data_dict[key] = val[keep]
trunc_data_dict["terminals"] = new_terminal[keep]
trunc_data_dict["timeouts"] = new_timeout[keep]


Checking how much is left

In [12]:
term = trunc_data_dict["terminals"].astype(bool)
tout = trunc_data_dict["timeouts"].astype(bool)

ends    = np.where(term | tout)[0]
starts  = np.concatenate([[0], ends[:-1] + 1])
lengths = ends - starts + 1

print(f"trajectories : {len(lengths):,}")
print(f"transitions  : {len(term):,}")
print(f"length mean  : {lengths.mean():.1f}")
print(f"length median: {np.median(lengths):.0f}")
print(f"length min/max: {lengths.min()} / {lengths.max()}")
print(f"clean survival rate: {np.sum(lengths == 1000)}")

N = 15   # your planning horizon
print(f"trajectories with length >= N: {(lengths >= N).mean():.1%}")

trajectories : 3,442
transitions  : 1,145,452
length mean  : 332.8
length median: 201
length min/max: 27 / 1000
clean survival rate: 329
trajectories with length >= N: 100.0%


Save Cleaned Dataset

In [13]:
save_filename = "FilteredSafetyPointGoal2Gymnasium_v0_200_3442.hdf5"

with h5py.File(save_filename, 'w') as f:
    for k, v in trunc_data_dict.items():
        f.create_dataset(name=k, data=v)
    f.attrs["source"]          = "SafetyPointGoal2Gymnasium_v0_200_3442.hdf5"
    f.attrs["transform"]       = "truncated at first violation (absorbing failure state)"
    f.attrs["cost_rule"]       = "c_t = 1[cost_t > 0]"
    f.attrs["n_trajectories"]  = 3442
    f.attrs["n_transitions"]   = len(trunc_data_dict["costs"])
    f.attrs["clean_survivors"] = 329